# Supply Chain Late Delivery Risk Predictor
**Capstone Project — Francis Amojelar**

Dataset: DataCo Smart Supply Chain for Big Data Analysis
Target: `Late_delivery_risk` (binary classification)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape)
print('\nTarget distribution:')
print(df['Late_delivery_risk'].value_counts(normalize=True).round(3))
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Late_delivery_risk'].value_counts().plot(kind='bar', ax=axes[0], color=['#01696f','#a12c7b'])
axes[0].set_title('Late Delivery Risk Distribution')
axes[0].set_xticklabels(['On Time (0)', 'Late (1)'], rotation=0)
df['Shipping_Mode'].value_counts().plot(kind='bar', ax=axes[1], color='#01696f')
axes[1].set_title('Orders by Shipping Mode')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
FEATURES = [
    'Type', 'Days_for_shipment_(scheduled)', 'Benefit_per_order',
    'Sales_per_customer', 'Category_Name', 'Customer_Segment',
    'Department_Name', 'Latitude', 'Longitude', 'Market',
    'Order_Item_Discount', 'Order_Item_Discount_Rate',
    'Order_Item_Product_Price', 'Order_Item_Profit_Ratio',
    'Order_Item_Quantity', 'Sales', 'Order_Item_Total',
    'Order_Profit_Per_Order', 'Product_Status', 'Shipping_Mode'
]
TARGET = 'Late_delivery_risk'

data = df[FEATURES + [TARGET]].copy()

# Encode categoricals
cat_cols = data.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    data[col] = data[col].fillna('Unknown')
    data[col] = le.fit_transform(data[col])

data = data.fillna(data.median(numeric_only=True))
print('Features ready:', data.shape)

## 4. Train / Test Split

In [ ]:
X = data.drop(columns=[TARGET])
y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', X_train.shape, '| Test size:', X_test.shape)

## 5. Model Training — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred, target_names=['On Time','Late']))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))

## 6. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.head(15).plot(kind='barh', color='#01696f')
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['On Time','Late'], yticklabels=['On Time','Late'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()